# GeoChat — Visual Question Answering

GeoChat can answer free-form natural language questions about satellite and aerial images. This notebook demonstrates:
1. How to format a VQA prompt for GeoChat
2. Asking different types of questions about the same image
3. Comparing GeoChat's responses to generic LLaVA-1.5 on the same RS images

## Prompt Format

GeoChat uses the LLaVA-1.5 instruction format:

```
A chat between a curious user and an artificial intelligence assistant specializing in remote sensing.
USER: <image>\n{question}
ASSISTANT:
```

## References
- GeoChat paper: https://arxiv.org/abs/2311.15826
- GeoChat model: https://huggingface.co/MBZUAI/GeoChat

## 1. Load Model (see 0-setup_and_model_loading.ipynb)

In [ ]:
import sys
import os
import torch
from PIL import Image
import urllib.request
from transformers import AutoTokenizer, AutoModelForCausalLM, CLIPImageProcessor, BitsAndBytesConfig

sys.path.insert(0, os.path.abspath("../../.."))
try:
    from dotenv_config import HF_TOKEN
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

model_id = "MBZUAI/GeoChat"

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN or None
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float32, token=HF_TOKEN or None
    )

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, token=HF_TOKEN or None)
image_processor = CLIPImageProcessor.from_pretrained(model_id, token=HF_TOKEN or None)
model.eval()
print("GeoChat loaded.")

## 2. Helper Function

In [ ]:
def ask_geochat(image_path, question, max_new_tokens=200):
    image = Image.open(image_path).convert("RGB")
    image_tensor = image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
    if torch.cuda.is_available():
        image_tensor = image_tensor.half().cuda()

    system_prompt = (
        "A chat between a curious user and an artificial intelligence assistant "
        "specializing in remote sensing."
    )
    prompt = f"{system_prompt} USER: <image>\n{question} ASSISTANT:"

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            images=image_tensor,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response.split("ASSISTANT:")[-1].strip()

## 3. Download Sample Remote Sensing Images

In [ ]:
import matplotlib.pyplot as plt

sample_images = {
    "airport": "https://upload.wikimedia.org/wikipedia/commons/thumb/9/9e/Linate_Airport_aerial.jpg/640px-Linate_Airport_aerial.jpg",
    "urban": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/ISS_view_of_Miami.jpg/640px-ISS_view_of_Miami.jpg",
    "farmland": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8c/Crop_circles_Aerial.jpg/640px-Crop_circles_Aerial.jpg",
}

os.makedirs("sample_images", exist_ok=True)
for name, url in sample_images.items():
    path = f"sample_images/{name}.jpg"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))
for ax, (name, _) in zip(axes, sample_images.items()):
    ax.imshow(plt.imread(f"sample_images/{name}.jpg"))
    ax.set_title(name.title())
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Ask Different Questions about the Same Image

In [ ]:
image_to_query = "sample_images/airport.jpg"

questions = [
    "Describe this aerial image in one sentence.",
    "What type of transportation infrastructure is visible?",
    "Approximately how many aircraft are visible on the tarmac?",
    "What is the dominant land cover type surrounding the main structure?",
]

print(f"Image: {os.path.basename(image_to_query)}")
print("=" * 60)
for q in questions:
    print(f"\nQ: {q}")
    response = ask_geochat(image_to_query, q)
    print(f"A: {response}")

## 5. Compare Across Different Scene Types

In [ ]:
question = "What is the primary land use or feature visible in this image?"

print(f"Question: '{question}'")
print("=" * 60)
for name in sample_images:
    image_path = f"sample_images/{name}.jpg"
    response = ask_geochat(image_path, question, max_new_tokens=100)
    print(f"\n[{name.upper()}]")
    print(f"  {response}")

## 6. Counting Objects

In [ ]:
counting_question = "Count and describe the man-made structures visible in this image."

for name in ["airport", "urban"]:
    image_path = f"sample_images/{name}.jpg"
    response = ask_geochat(image_path, counting_question, max_new_tokens=150)
    print(f"\n[{name.upper()}]")
    print(f"Q: {counting_question}")
    print(f"A: {response}")